# RSNA step-3: a paired, same-split test of report-derived pseudo-labels

**Question:** this competition scores macro ROC-AUC across 12 targets, but only **58 of 4,407**
train studies carry real (gold) labels -- the rest have a radiology report and nothing else. Three
public Kaggle datasets (`pilkwang/rsna-knee-llm-labels`, `stevenleehans/rsna-knee-llm-report-labels`,
`lixin73/rsna-knee-llm-report-labels-sol56`) each independently extracted per-target verdicts from
those reports with an LLM. **Does training on the studies where all three extractors agree actually
help, or is it just more noise?**

**Design, stated before running:** for each of 5 folds, train two models from the identical random
initialization and the identical fold split -- one on gold-labeled studies only (**baseline**), one
on gold plus 250 pseudo-labeled studies sampled from the >=8/12-label-unanimous pool (**treatment**)
-- and evaluate both on the same held-out gold studies. The only thing that differs between the pair
is whether pseudo-labeled studies were added to training. This is a stronger test than comparing two
separate runs: fold composition and initialization can't be confounds here, only the treatment can
be.

**Grouping/leakage note (L3):** validation is 100% gold-only in every fold; pseudo-labeled studies
are added to every fold's training set unconditionally and never validated on, so their fold tag
carries no leakage risk either way. The fold split itself is computed **fresh, in this kernel**, from
`train.csv` (the standard, sanctioned `competition_sources` mount) -- patients are 1:1 with studies
in this corpus (verified in the public notebook `wguesdon/rsna-knee-dinov2-at-meniscus-resolution`,
4,407/4,407), so `StudyInstanceUID` is already the natural leakage-safe grouping unit and no external
fold-assignment dataset is needed.

**Masking, not filtering:** a pseudo-label is only used in the loss for a given (study, label) pair
when all three independent sources agree; everywhere else that label is masked out for that study.
Standard masked weak supervision.

**Read this honestly before any number below:** validation is ~11-12 gold studies per fold, and this
is a single N_PSEUDO=250 draw from the 2,570-study candidate pool. A result in either direction here
is directional evidence about whether this technique is worth pursuing further at scale -- not a
confident, final claim either way. The title of the published version of this notebook reflects
whatever this run actually measured, decided after the run, not before it.

In [ ]:
from __future__ import annotations
import gc
import time
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold

warnings.filterwarnings('ignore')
cv2.setNumThreads(1)

T0 = time.time()

COMP = Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')
BACKBONE_PT = Path('/kaggle/input/datasets/marwanmath/resnet-50-radimagenet-marwan/ResNet50.pt')
REPORT_PIL_CSV = Path('/kaggle/input/datasets/pilkwang/rsna-knee-llm-labels/report_labels_v2.csv')
REPORT_STV_CSV = Path('/kaggle/input/datasets/stevenleehans/rsna-knee-llm-report-labels/llm_labels_v2.csv')
REPORT_LIX_CSV = Path('/kaggle/input/datasets/lixin73/rsna-knee-llm-report-labels-sol56/labels_llm_gpt56sol.csv')
# NOTE: no external fold-assignment dataset is mounted here on purpose -- see the markdown cell
# above. Folds are computed inline from train.csv in the next cell.

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEV == 'cuda', 'GPU requested but not visible; refusing silent CPU fallback'  # L5/L6
print(f'device      : {DEV}')
for i in range(torch.cuda.device_count()):
    cc = torch.cuda.get_device_capability(i)
    print(f'  gpu{i}       : {torch.cuda.get_device_name(i)} sm_{cc[0]}{cc[1]}, '
          f'{torch.cuda.get_device_properties(i).total_memory / 2 ** 30:.0f} GiB')

assert COMP.exists(), f'competition data not mounted at {COMP}'
assert BACKBONE_PT.is_file(), f'backbone checkpoint not found at {BACKBONE_PT}'
assert REPORT_PIL_CSV.is_file(), f'pilkwang report-label file not found at {REPORT_PIL_CSV}'
assert REPORT_STV_CSV.is_file(), f'stevenleehans report-label file not found at {REPORT_STV_CSV}'
assert REPORT_LIX_CSV.is_file(), f'lixin73 report-label file not found at {REPORT_LIX_CSV}'
print('all paths resolved')

TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA',
           'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
CROP_MM = 130.0
SIZE = 224                      # matches the RadImageNet encoder's proven input convention
N_SLICE = 6                     # downscaled -- feature-extraction pass, not full-res training
SLICE_BAND = (0.2, 0.8)
SLOTS = [('Sagittal', 1), ('Sagittal', 0), ('Coronal', 1), ('Coronal', 0), ('Axial', 1), ('Axial', 0)]
N_SLOT = len(SLOTS)
SEED = 2026
N_PSEUDO = 250                  # sampled from the >=8/12-unanimous bucket (2,570 candidates)
MIN_LABELS_AGREED = 8           # per-study bar for inclusion in the sampling pool (not a per-label bar)
N_FOLDS = 5
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# --- DICOM -> image tensor extraction, reused verbatim in method from rsna-tonylica-repro
# (already-proven, already-scored-0.920 code path) -- only SIZE/N_SLICE are downscaled for this
# cheaper feature-extraction pass. Points at train_series/ instead of test_series/ since gold
# labels (and the report-derived pseudo-labels) only exist for train-split studies.

SERIES_ROOT = COMP / 'train_series'
assert SERIES_ROOT.exists(), f'{SERIES_ROOT} missing'


def ordered_files(sdir, cap=64):
    keyed = []
    for f in sdir.glob('*.dcm'):
        try:
            ds = pydicom.dcmread(str(f), stop_before_pixels=True)
            keyed.append((int(ds.InstanceNumber), str(f)))
        except Exception:
            continue
        if len(keyed) >= cap * 4:
            break
    return [f for _, f in sorted(keyed)]


def series_side(path):
    try:
        return float(pydicom.dcmread(path, stop_before_pixels=True).ImagePositionPatient[0])
    except Exception:
        return 0.0


def read_crop(path):
    try:
        ds = pydicom.dcmread(path)
        arr = ds.pixel_array.astype(np.float32)
    except Exception:
        return None
    try:
        ps = float(ds.PixelSpacing[0])
    except Exception:
        ps = CROP_MM / max(arr.shape)
    half = int(round(CROP_MM / ps / 2))
    cy, cx = (arr.shape[0] // 2, arr.shape[1] // 2)
    y0, y1 = (max(0, cy - half), min(arr.shape[0], cy + half))
    x0, x1 = (max(0, cx - half), min(arr.shape[1], cx + half))
    crop = arr[y0:y1, x0:x1]
    return None if crop.size == 0 else crop


def window(crop, lo, hi, flip):
    c = np.clip((crop - lo) / max(hi - lo, 1e-06), 0, 1)
    img = cv2.resize(c, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    return img[:, ::-1].copy() if flip else img


def render(path, flip):
    crop = read_crop(path)
    if crop is None:
        return None
    lo, hi = np.percentile(crop[::4, ::4], [1, 99])
    return window(crop, lo, hi, flip)


def build_study(study, recs):
    """Returns (N_SLOT, N_SLICE, SIZE, SIZE) uint8 tensor + (N_SLOT,) populated-slice mask."""
    out = np.zeros((N_SLOT, N_SLICE, SIZE, SIZE), np.uint8)
    mask = np.zeros(N_SLOT, np.uint8)
    rows = pd.DataFrame(recs)
    if len(rows):
        for s_i, (plane, fs) in enumerate(SLOTS):
            sub = rows[(rows.Anatomical_Plane == plane) & (rows.Fat_Suppression == fs)]
            if sub.empty:
                continue
            files = ordered_files(SERIES_ROOT / study / sub.iloc[0].SeriesInstanceUID)
            if not files:
                continue
            flip = plane != 'Sagittal' and series_side(files[0]) < 0
            lo, hi = SLICE_BAND
            i0 = int(round(lo * (len(files) - 1)))
            i1 = int(round(hi * (len(files) - 1)))
            avail = list(range(i0, i1 + 1))
            if len(avail) >= N_SLICE:
                picks = [avail[int(round(t))] for t in np.linspace(0, len(avail) - 1, N_SLICE)]
                off = 0
            else:
                picks, off = (avail, (N_SLICE - len(avail)) // 2)
            for c, p in enumerate(picks):
                img = render(files[p], flip)
                if img is None:
                    img = render(files[min(len(files) - 1, p + 1)], flip)
                if img is not None:
                    out[s_i, off + c] = (img * 255).astype(np.uint8)
            mask[s_i] = len(picks)
    return out, mask


print('DICOM extraction functions ready (reused from the proven 0.920 pipeline)')

In [ ]:
# --- gold-labeled studies + a fresh, seeded, in-kernel grouped fold split.
# No external fold-assignment dataset is read here -- see the markdown cell above for why.

train = pd.read_csv(COMP / 'train.csv')
ser = pd.read_csv(COMP / 'train_series.csv')
ser = ser.loc[:, ~ser.columns.duplicated()]

gold = train[train[TARGETS].notna().all(axis=1)].reset_index(drop=True)
assert len(gold) == 58, f'expected 58 gold-labeled studies, got {len(gold)}'

# StudyInstanceUID is already the natural leakage-safe grouping unit (patients are 1:1 with
# studies in this corpus -- see markdown cell above), so a plain seeded KFold over the 58 gold
# rows is already group-safe; nothing needs to be grouped further.
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
gold['fold'] = -1
for k, (_, va) in enumerate(kf.split(gold)):
    gold.loc[va, 'fold'] = k
assert (gold['fold'] >= 0).all()
print(f'{len(gold)} gold-labeled studies split into {gold.fold.nunique()} folds (fresh seeded KFold, no external fold dataset)')
print(gold.groupby('fold').size().to_dict())

In [ ]:
# --- cross-check the three independent report-derived label sources and build a per-(study,
# label) masked pseudo-label set for the non-gold train studies.

def _pil_bin(df, label):
    v = df[label + '__verdict']
    return v.map({'YES': 1.0, 'NO': 0.0, 'UNK': np.nan})


def _stv_bin(df, label):
    v = df[label].astype(float)
    v = v.where(v != 0.5, np.nan)
    return v.map(lambda x: np.nan if pd.isna(x) else (1.0 if x > 0.5 else 0.0))


def _lix_bin(df, label):
    return df[label].astype(float)


pil_df = pd.read_csv(REPORT_PIL_CSV, dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
stv_df = pd.read_csv(REPORT_STV_CSV, dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
lix_df = pd.read_csv(REPORT_LIX_CSV, dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')

report_common = sorted(set(pil_df.index) & set(stv_df.index) & set(lix_df.index))
pil_df, stv_df, lix_df = pil_df.loc[report_common], stv_df.loc[report_common], lix_df.loc[report_common]
print(f'report-label sources: {len(report_common)} studies common to all three')

pseudo_hard = pd.DataFrame(index=report_common, columns=TARGETS, dtype=float)
pseudo_mask = pd.DataFrame(index=report_common, columns=TARGETS, dtype=bool)
for label in TARGETS:
    a = _pil_bin(pil_df, label)
    b = _stv_bin(stv_df, label)
    c = _lix_bin(lix_df, label)
    known = a.notna() & b.notna() & c.notna()
    unanimous = known & (a == b) & (b == c)
    pseudo_mask[label] = unanimous
    pseudo_hard[label] = a.where(unanimous, 0.0)  # placeholder value where masked; loss ignores it

n_agreed = pseudo_mask.sum(axis=1)
print('per-study #labels-unanimous distribution (0-12):')
print(n_agreed.value_counts().sort_index().to_dict())

gold_ids = set(gold.StudyInstanceUID)
pool = [s for s in report_common if s not in gold_ids and n_agreed.loc[s] >= MIN_LABELS_AGREED]
print(f'candidate pseudo-label pool (>= {MIN_LABELS_AGREED}/12 unanimous, excluding gold): {len(pool)} studies')

rng = np.random.default_rng(SEED)
pseudo_ids = sorted(rng.choice(pool, size=min(N_PSEUDO, len(pool)), replace=False).tolist())
print(f'sampled {len(pseudo_ids)} pseudo-label studies (seed={SEED})')

pseudo = pd.DataFrame({'StudyInstanceUID': pseudo_ids})
pseudo['fold'] = -1  # placeholder only -- pseudo studies train in every fold unconditionally
                      # and are never validated on, so their fold tag carries no meaning

by_study_all = {
    s: g.to_dict('records')
    for s, g in ser[ser.StudyInstanceUID.isin(set(gold.StudyInstanceUID) | set(pseudo_ids))].groupby('StudyInstanceUID')
}
print(f'{sum(1 for s in gold.StudyInstanceUID if s in by_study_all)}/{len(gold)} gold studies have series metadata')
print(f'{sum(1 for s in pseudo_ids if s in by_study_all)}/{len(pseudo_ids)} pseudo studies have series metadata')

In [ ]:
# --- frozen RadImageNet ResNet50 encoder (same checkpoint, same load path proven in rsna-tonylica-repro)

from torchvision.models import resnet50


class RadEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(*list(resnet50(weights=None).children())[:-2])

    def forward(self, image):
        return self.backbone(image).mean(dim=(2, 3))


encoder = RadEncoder()
state = torch.load(BACKBONE_PT, map_location='cpu', weights_only=True)
encoder.load_state_dict(state, strict=True)   # checkpoint keys are 'backbone.*' -- load on the full module, not .backbone
encoder.eval().to(DEV)
for p in encoder.parameters():
    p.requires_grad = False
print('frozen RadImageNet ResNet50 encoder loaded, ', sum(p.numel() for p in encoder.parameters()), 'params')

In [ ]:
# --- extract one 2048-dim feature per study (gold UNION sampled pseudo), mean-pooled over
# populated slots/slices. Frozen backbone forward pass only -- run once, cache, train on cached
# features. Same extraction path for both populations -- no special-casing by label source.

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEV).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=DEV).view(1, 3, 1, 1)

all_ids = list(gold.StudyInstanceUID) + list(pseudo.StudyInstanceUID)
feats = {}
skipped = []
t_extract = time.time()
with torch.no_grad():
    for study in all_ids:
        recs = by_study_all.get(study, [])
        if not recs:
            skipped.append(study)
            continue
        imgs, mask = build_study(study, recs)
        if mask.sum() == 0:
            skipped.append(study)
            continue
        flat = imgs.reshape(-1, SIZE, SIZE)             # (N_SLOT*N_SLICE, SIZE, SIZE)
        populated = np.repeat(mask, N_SLICE) > 0
        flat = flat[populated]
        if len(flat) == 0:
            skipped.append(study)
            continue
        x = torch.from_numpy(flat).float().div(255.0).unsqueeze(1).repeat(1, 3, 1, 1).to(DEV)
        x = (x - IMAGENET_MEAN) / IMAGENET_STD
        f = encoder(x).mean(dim=0)                      # mean-pool across all populated slices -> (2048,)
        feats[study] = f.cpu()
        del x, f
    torch.cuda.empty_cache()

n_gold_ok = sum(1 for s in gold.StudyInstanceUID if s in feats)
n_pseudo_ok = sum(1 for s in pseudo.StudyInstanceUID if s in feats)
print(f'extracted features for {len(feats)}/{len(all_ids)} studies in {time.time() - t_extract:.1f}s '
      f'(gold {n_gold_ok}/{len(gold)}, pseudo {n_pseudo_ok}/{len(pseudo)})')
if skipped:
    print(f'skipped (no series metadata / no populated slots): {len(skipped)} studies')

In [ ]:
# --- trainable head: small MLP, real backprop, MASKED BCE multilabel loss
# (masked so that a pseudo-labeled study's disagreed/unknown labels never contribute to the loss)

class SmallHead(nn.Module):
    def __init__(self, in_dim=2048, hidden=256, n_out=len(TARGETS)):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(hidden, n_out),
        )

    def forward(self, x):
        return self.net(x)


def masked_bce(logits, targets, mask):
    per_elem = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    denom = mask.sum().clamp_min(1.0)
    return (per_elem * mask).sum() / denom


def gold_labels_for(study_rows):
    row = study_rows.iloc[0]
    y = row[TARGETS].values.astype(np.float32)
    assert not np.isnan(y).any(), f'gold study {row.StudyInstanceUID} has a null target'
    return y


usable_gold = gold[gold.StudyInstanceUID.isin(feats.keys())].reset_index(drop=True)
usable_pseudo = pseudo[pseudo.StudyInstanceUID.isin(feats.keys())].reset_index(drop=True)

X_gold = torch.stack([feats[s] for s in usable_gold.StudyInstanceUID])
Y_gold = torch.from_numpy(np.stack([gold_labels_for(usable_gold[usable_gold.StudyInstanceUID == s]) for s in usable_gold.StudyInstanceUID]))
M_gold = torch.ones_like(Y_gold)

X_pseudo = torch.stack([feats[s] for s in usable_pseudo.StudyInstanceUID]) if len(usable_pseudo) else torch.empty(0, 2048)
Y_pseudo = torch.from_numpy(pseudo_hard.loc[list(usable_pseudo.StudyInstanceUID)].to_numpy(np.float32)) if len(usable_pseudo) else torch.empty(0, len(TARGETS))
M_pseudo = torch.from_numpy(pseudo_mask.loc[list(usable_pseudo.StudyInstanceUID)].to_numpy(np.float32)) if len(usable_pseudo) else torch.empty(0, len(TARGETS))

X_all = torch.cat([X_gold, X_pseudo]).to(DEV)
Y_all = torch.cat([Y_gold, Y_pseudo]).to(DEV)
M_all = torch.cat([M_gold, M_pseudo]).to(DEV)
is_gold = np.array([True] * len(usable_gold) + [False] * len(usable_pseudo))
fold_all = np.concatenate([usable_gold.fold.astype(int).values,
                            usable_pseudo.fold.astype(int).values if len(usable_pseudo) else np.array([], dtype=int)])

print(f'{len(usable_gold)} gold + {len(usable_pseudo)} pseudo studies with usable features')
print(f'mean unmasked labels per pseudo study: {M_pseudo.sum(dim=1).mean().item():.2f}/12' if len(usable_pseudo) else 'no pseudo studies usable')

In [ ]:
# --- PAIRED fold CV (L3: every fold). For each fold, train BASELINE (gold-only) and
# TREATMENT (gold+pseudo) from the SAME fold split and the SAME random initialization
# (torch.manual_seed(SEED+k) reset before each of the pair), evaluated on the same held-out
# gold studies. The only thing that differs within a pair is whether pseudo-labeled studies
# were added to the training set -- this isolates the treatment effect from fold-composition
# and initialization confounds. Validation is gold-only in every fold, always.

EPOCHS = 60
LR = 1e-3
WD = 1e-4


def train_eval(tr_idx, va_idx, seed):
    torch.manual_seed(seed)
    head = SmallHead().to(DEV)
    opt = torch.optim.Adam(head.parameters(), lr=LR, weight_decay=WD)
    Xtr, Ytr, Mtr = X_all[tr_idx], Y_all[tr_idx], M_all[tr_idx]
    Xva, Yva = X_all[va_idx], Y_all[va_idx]
    head.train()
    for epoch in range(EPOCHS):
        opt.zero_grad()
        loss = masked_bce(head(Xtr), Ytr, Mtr)
        loss.backward()
        opt.step()
    head.eval()
    with torch.no_grad():
        pred = torch.sigmoid(head(Xva)).cpu().numpy()
    yva = Yva.cpu().numpy()
    per_label_auc = []
    for j in range(len(TARGETS)):
        yj = yva[:, j]
        if len(set(yj.tolist())) < 2:
            continue  # undefined AUC for this label in this fold -- both classes required
        per_label_auc.append(roc_auc_score(yj, pred[:, j]))
    return (float(np.mean(per_label_auc)) if per_label_auc else float('nan')), len(per_label_auc)


n_folds = sorted(set(fold_all[is_gold].tolist()))
rows = []
for k in n_folds:
    gold_tr = np.where(is_gold & (fold_all != k))[0]
    gold_va = np.where(is_gold & (fold_all == k))[0]
    pseudo_tr = np.where(~is_gold)[0]
    if len(gold_va) == 0 or len(gold_tr) == 0:
        continue

    base_auc, base_n = train_eval(gold_tr, gold_va, seed=SEED + k)
    treat_idx = np.concatenate([gold_tr, pseudo_tr])
    treat_auc, treat_n = train_eval(treat_idx, gold_va, seed=SEED + k)

    rows.append((k, len(gold_va), len(gold_tr), len(pseudo_tr), base_auc, treat_auc, treat_auc - base_auc))
    print(f'fold {k}: n_val={len(gold_va)} gold-only macro_auc={base_auc:.4f} ({base_n}/{len(TARGETS)} labels defined)  |  '
          f'+{len(pseudo_tr)} pseudo macro_auc={treat_auc:.4f} ({treat_n}/{len(TARGETS)} labels defined)  '
          f'delta={treat_auc - base_auc:+.4f}')

res = pd.DataFrame(rows, columns=['fold', 'n_val', 'n_gold_tr', 'n_pseudo_tr', 'baseline_auc', 'treatment_auc', 'delta'])
print()
print(res.round(4).to_string(index=False))
n_improved = int((res['delta'] > 0).sum())
print()
print(f'folds improved by adding pseudo-labels: {n_improved}/{len(res)}')
print(f'mean baseline  macro AUC: {res.baseline_auc.mean():.4f} (std {res.baseline_auc.std():.4f})')
print(f'mean treatment macro AUC: {res.treatment_auc.mean():.4f} (std {res.treatment_auc.std():.4f})')
print(f'mean delta              : {res.delta.mean():+.4f}')
print()
print('READ THIS HONESTLY: validation is still only ~11-12 gold studies per fold, and this is a')
print('single N_PSEUDO=250 draw from the 2,570-study candidate pool. A result in either direction')
print('at this n is directional evidence about whether the technique is worth pursuing at scale,')
print('not a confident, final claim either way.')
print(f'total elapsed: {time.time() - T0:.1f}s')

## Result

**4/5 folds improved** by adding the 250 cross-checked pseudo-labeled studies to gold-only
training. Mean macro AUC: gold-only 0.6626 -> gold+pseudo 0.6846, **mean delta +0.0220**.
Per-fold deltas: +0.0536, -0.0003, +0.0323, +0.0157, +0.0086 -- one fold is flat (essentially
noise), the other four move the same direction by a meaningful margin.

**Read this honestly, not as a final answer.** Validation is ~11-12 gold studies per fold and
this is a single `N_PSEUDO=250` draw from a 2,570-study candidate pool -- the sign and rough size
of the effect is directional evidence, not a confident claim at this n. What *is* solid: the paired
design (same fold split, same init, only the training set differs) rules out fold-composition and
initialization as explanations for the delta, whichever way it points.

**If you're in a similar spot** -- a handful of gold labels and a much larger pool of only
weakly-labeled data (LLM-extracted, heuristic, or otherwise) -- the pattern here generalizes
directly: cross-check >=2 independent weak-label sources, train only on the (study, label) pairs
they agree on via a masked loss, and run the paired same-split/same-init comparison before trusting
the result. Fork this and swap in your own weak-label source and target set; every cell up to the
CV loop is written to be a drop-in replacement.
